 Notebook: erp_ingest_sales_orders
 Layer: Ingestion → Bronze
 Source: ERP (CSV files from Unity Catalog Volume)

 Description:
 - Validates raw data (batch - commented)
 - Ingests data using Auto Loader (incremental)
 - Writes to Bronze Delta table

 Architecture:
 Volumes → Auto Loader → Bronze



 Step 2 (OPTIONAL): Batch read for validation
 DO NOT USE IN PIPELINE (kept for reference)



  Define source path (Unity Catalog Volume)

 This path contains multiple CSV files (ERP sales orders)


In [0]:
source_path = "/Volumes/workspace/default/raw_data/erp/sales_orders/"


 Read raw CSV files from volume
#
 - Reads all files in folder
 - Uses correct delimiter (;)
 - No transformations (raw ingestion)



In [0]:
df_sales_orders = spark.read \
    .format("csv") \
    .option("inferschema",True)\
    .option("header", "true") \
    .option("delimiter", ";") \
    .load(source_path)


  Validate data


In [0]:
df_sales_orders.printSchema()

display(df_sales_orders)

# --------------------------------------------
#  Row count check
# --------------------------------------------

In [0]:
df_sales_orders.count()


 Step 3: Define source and checkpoint paths


In [0]:
source_path = "/Volumes/workspace/default/raw_data/erp/sales_orders/"
checkpoint_path = "/Volumes/workspace/default/raw_data/checkpoints/sales_orders_checkpoint"


 Step 4: Define Schema


In [0]:
import pyspark.sql.types as T

sales_orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("order_date", T.StringType(), True),  
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True),
    T.StructField("total_amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True)
])


 Step 5: Read sales_orders using Auto Loader (with metadata)
 - Incremental ingestion
 - Detects new files automatically
 - Adds ingestion metadata (timestamp + source file)
 - Handles schema evolution


In [0]:
import pyspark.sql.functions as F

schema_location = "/Volumes/workspace/default/raw_data/schemas/sales_orders"  

df_sales_orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("cloudFiles.schemaLocation", schema_location)        
    .option("cloudFiles.schemaHints",                           
        "order_id STRING, order_date STRING, customer_id STRING, "
        "product_id STRING, quantity INT, unit_price DOUBLE, "
        "total_amount DOUBLE, status STRING")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")   
    .load(source_path)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)


 Step 6: Write stream to Bronze Delta table
 - Stores raw ERP sales data
 - Append-only (no transformations)
 - Checkpoint ensures incremental processing


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
(
    df_sales_orders_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze.erp_sales_orders")
)


 Step 7: Validate Bronze table
 - Verify ingestion worked
 - Check metadata columns


In [0]:
display(spark.table("bronze.erp_sales_orders"))

In [0]:
%sql
Select 
count(*)
from bronze.erp_sales_orders